# LeetCode 3852 — Smallest Pair With Different Frequencies bằng Python

Link đề: https://leetcode.com/problems/smallest-pair-with-different-frequencies/ · Độ khó: **Easy** · Chủ đề: Array, Hash Table, Counting

Skeleton LeetCode:

```python
class Solution:
    def minDistinctFreqPair(self, nums: List[int]) -> List[int]:
        ...
```

> Ghi chú: tên bài, ví dụ, constraints và chữ ký hàm đã được đối chiếu qua leetcode.doocs.org (Easy, không Premium); phần phát biểu đề được viết lại bằng lời của mình nên nên đối chiếu từng chữ với đề gốc nếu cần.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: tìm cặp **giá trị** `(x, y)` với `x < y` mà `x` và `y` xuất hiện trong `nums` với **số lần khác nhau**; chọn cặp có `x` nhỏ nhất, rồi `y` nhỏ nhất.
2. Biết đếm tần suất bằng `Counter` hoặc mảng đếm kích thước 101.
3. Nhận ra **so sánh từ điển** (lexicographic): chọn `x` nhỏ nhất trước rồi mới tới `y`.
4. Rút ra insight: nếu có cặp hợp lệ thì `x` chính là **giá trị nhỏ nhất** của mảng; từ đó có lời giải `O(n)`.
5. Viết 5 cách khác nhau và tránh các lỗi: so sánh nhầm giá trị thay vì tần suất, chỉ xét cặp kề nhau, trả về `[y, x]`, quên `[-1, -1]`.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums`. Xét các cặp **giá trị phân biệt** `(x, y)` thỏa:

- `x < y`, và
- số lần `x` xuất hiện trong `nums` **khác** số lần `y` xuất hiện.

Trong các cặp hợp lệ, chọn cặp có `x` **nhỏ nhất**; nếu còn nhiều cặp cùng `x`, chọn cặp có `y` **nhỏ nhất**. Trả về `[x, y]`. Nếu không tồn tại cặp nào, trả về `[-1, -1]`.

| Ví dụ | nums | Output | Giải thích |
|---|---|---|---|
| 1 | `[1,1,2,2,3,4]` | `[1,3]` | `1` xuất hiện 2 lần; giá trị nhỏ nhất lớn hơn 1 mà tần suất khác 2 là `3` (xuất hiện 1 lần) |
| 2 | `[1,5]` | `[-1,-1]` | Cả hai xuất hiện 1 lần |
| 3 | `[7]` | `[-1,-1]` | Chỉ có một giá trị phân biệt |

Constraints: `1 <= nums.length <= 100`; `1 <= nums[i] <= 100`.

In [1]:
from collections import Counter
for nums in ([1, 1, 2, 2, 3, 4], [1, 5], [7]):
    print(nums, dict(sorted(Counter(nums).items())))

[1, 1, 2, 2, 3, 4] {1: 2, 2: 2, 3: 1, 4: 1}
[1, 5] {1: 1, 5: 1}
[7] {7: 1}


## 3. Kiến thức nền

### 3.1. `Counter` đếm tần suất

`Counter(nums)` trả về từ điển giá trị -> số lần xuất hiện. Truy cập `cnt[v]` cho giá trị không có sẽ trả `0` (không lỗi).

In [2]:
from collections import Counter
cnt = Counter([1, 1, 2, 2, 3, 4])
print(cnt, cnt[1], cnt[3], cnt[99])

Counter({1: 2, 2: 2, 3: 1, 4: 1}) 2 1 0


### 3.2. `sorted(set(...))` và `sorted(cnt)`

`set(nums)` bỏ trùng, `sorted(...)` sắp xếp tăng dần. Duyệt trực tiếp `Counter` (hay `dict`) cho thứ tự xuất hiện chứ không phải thứ tự tăng, nên cần `sorted`.

In [3]:
nums = [4, 1, 1, 3, 2, 2]
print(set(nums))
print(sorted(set(nums)))
print(list(Counter(nums)))          # thứ tự xuất hiện
print(sorted(Counter(nums)))        # thứ tự tăng dần

{1, 2, 3, 4}
[1, 2, 3, 4]
[4, 1, 3, 2]
[1, 2, 3, 4]


### 3.3. `itertools.combinations` và thứ tự từ điển

`combinations(sorted_list, 2)` sinh các cặp `(a, b)` với `a` đứng trước `b` trong list, **theo thứ tự từ điển** (cặp `(1,2), (1,3), ..., (2,3), ...`). Với list đã sắp xếp, cặp đầu tiên thỏa điều kiện chính là cặp có `x` nhỏ nhất rồi `y` nhỏ nhất.

In [4]:
from itertools import combinations
print(list(combinations([1, 2, 3, 4], 2)))

[(1, 2), (1, 3), (1, 4), (2, 3), (2, 4), (3, 4)]


### 3.4. So sánh tuple và hàm `min`

Python so sánh tuple theo từ điển: `(1, 3) < (2, 2)` là `True` vì `1 < 2`; `(1, 3) < (1, 4)` vì `3 < 4`. Vì vậy `min(danh_sách_các_cặp)` tự động chọn đúng cặp "x nhỏ nhất, rồi y nhỏ nhất".

In [5]:
pairs = [(2, 3), (1, 4), (1, 3), (2, 4)]
print(min(pairs), (1, 3) < (2, 2), (1, 3) < (1, 4))
print(min([], default=None))

(1, 3) True True
None


### 3.5. Mảng đếm (counting array)

Vì `nums[i] <= 100`, ta có thể dùng mảng `freq` dài `101`: `freq[v] += 1`. Truy cập `freq[v]` là `O(1)`, không cần `dict`.

In [6]:
freq = [0] * 101
for v in [1, 1, 2, 2, 3, 4]:
    freq[v] += 1
print(freq[:6], [v for v in range(1, 101) if freq[v]])

[0, 2, 2, 1, 1, 0] [1, 2, 3, 4]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Chuyển đề về hai bước:

1. Gom lại thành bảng **giá trị -> tần suất** (mỗi giá trị phân biệt chỉ còn một dòng).
2. Trong các giá trị đã sắp xếp `v0 < v1 < v2 < ...` tìm cặp `(vi, vj)` với `i < j` mà tần suất khác nhau, chọn cặp nhỏ nhất theo thứ tự từ điển.

Tính tay `[1,1,2,2,3,4]`:

| giá trị | 1 | 2 | 3 | 4 |
|---|---|---|---|---|
| tần suất | 2 | 2 | 1 | 1 |

- `x = 1` (nhỏ nhất): thử `y = 2` (2 = 2, không); `y = 3` (2 != 1, **được**) -> cặp `(1, 3)`. Vì `x = 1` đã nhỏ nhất có thể, ta dừng.

Nếu `x = 1` không có `y` hợp lệ (mọi giá trị khác đều cùng tần suất với 1) thì ta thử `x = 2`... nhưng khi đó `x = 2` cũng có cùng tần suất với tất cả -> vô nghiệm. Đây là gợi ý cho insight ở mục 6.

In [7]:
cnt = Counter([1, 1, 2, 2, 3, 4])
vals = sorted(cnt)
for x in vals:
    for y in vals:
        if y > x:
            print(f"({x},{y}): freq {cnt[x]} vs {cnt[y]} -> {'hợp lệ' if cnt[x] != cnt[y] else 'không'}")

(1,2): freq 2 vs 2 -> không
(1,3): freq 2 vs 1 -> hợp lệ
(1,4): freq 2 vs 1 -> hợp lệ
(2,3): freq 2 vs 1 -> hợp lệ
(2,4): freq 2 vs 1 -> hợp lệ
(3,4): freq 1 vs 1 -> không


## 5. Cách 1 — Brute Force / baseline

Với `x` tăng dần từ `1` đến `100`, nếu `x` có trong mảng thì với `y` tăng dần từ `x + 1` đến `100`, nếu `y` có trong mảng và `nums.count(x) != nums.count(y)` thì trả về `[x, y]` ngay (cặp đầu tiên gặp phải chính là cặp nhỏ nhất). Nếu không có cặp nào thì `[-1, -1]`.

Nút thắt: `nums.count(...)` mỗi lần quét cả mảng `O(n)`, và có `O(100²)` cặp -> `O(100² · n)` (tối đa ~10^6 phép so sánh, vẫn chấp nhận được với `n <= 100`) nhưng rất lãng phí vì ta đếm lại cùng một giá trị nhiều lần.

In [8]:
class SolutionBrute:
    def minDistinctFreqPair(self, nums):
        for x in range(1, 101):
            if x not in nums:
                continue
            for y in range(x + 1, 101):
                if y in nums and nums.count(x) != nums.count(y):
                    return [x, y]
        return [-1, -1]

print(SolutionBrute().minDistinctFreqPair([1, 1, 2, 2, 3, 4]), SolutionBrute().minDistinctFreqPair([1, 5]), SolutionBrute().minDistinctFreqPair([7]))

[1, 3] [-1, -1] [-1, -1]


**Độ phức tạp:** Time `O(V² · n)` với `V = 100` là khoảng giá trị (hằng số), Space `O(1)`.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** gọi `v0` là **giá trị nhỏ nhất** trong `nums` và `f0 = tần suất của v0`.

- Nếu **mọi** giá trị phân biệt đều có tần suất `f0` thì không có cặp nào hợp lệ -> `[-1, -1]`.
- Ngược lại, tồn tại giá trị `v` với tần suất `!= f0`; khi đó `(v0, v)` là cặp hợp lệ. Vì `v0` là giá trị nhỏ nhất của mảng nên **không thể có `x` nào nhỏ hơn `v0`** -> `x = v0`. Còn `y` là **giá trị nhỏ nhất có tần suất khác `f0`**.

Vậy chỉ cần: sắp xếp các giá trị phân biệt, lấy `x = v0`, quét tìm `y` đầu tiên có tần suất khác `f0`. Chỉ một lượt duyệt: `O(m)`.

Invariant: khi duyệt các giá trị theo thứ tự tăng, mọi giá trị đã duyệt trước `y` đều có tần suất đúng bằng `f0` (nên chúng không thể làm `y` tốt hơn).

## 7. Các cách tiếp theo

### Cách 2 — `Counter` + `combinations` trên giá trị đã sắp xếp

Đếm bằng `Counter` (mỗi giá trị chỉ đếm một lần), sắp xếp các giá trị phân biệt rồi duyệt các cặp theo thứ tự từ điển; cặp đầu tiên có tần suất khác nhau là đáp án.

In [9]:
from collections import Counter
from itertools import combinations

class SolutionCombos:
    def minDistinctFreqPair(self, nums):
        cnt = Counter(nums)
        for x, y in combinations(sorted(cnt), 2):
            if cnt[x] != cnt[y]:
                return [x, y]
        return [-1, -1]

print(SolutionCombos().minDistinctFreqPair([1, 1, 2, 2, 3, 4]), SolutionCombos().minDistinctFreqPair([1, 5]))

[1, 3] [-1, -1]


**Độ phức tạp:** Time `O(n + m²)` (số cặp `m(m-1)/2`), Space `O(m)`, `m` = số giá trị phân biệt (`m <= 100`).

### Cách 3 — Gom mọi cặp hợp lệ rồi lấy `min`

Duyệt hết các cặp, đưa các cặp hợp lệ vào list rồi lấy `min` theo thứ tự tuple. Cách này không phụ thuộc vào thứ tự duyệt nên dễ kiểm chứng đúng sai (làm oracle).

In [10]:
class SolutionMinTuple:
    def minDistinctFreqPair(self, nums):
        cnt = Counter(nums)
        pairs = [(x, y) for x in cnt for y in cnt if x < y and cnt[x] != cnt[y]]
        return list(min(pairs)) if pairs else [-1, -1]

print(SolutionMinTuple().minDistinctFreqPair([1, 1, 2, 2, 3, 4]), SolutionMinTuple().minDistinctFreqPair([7]))

[1, 3] [-1, -1]


**Độ phức tạp:** Time `O(n + m²)`, Space `O(m²)` (list các cặp).

### Cách 4 — Insight: so với giá trị nhỏ nhất (`O(m)`)

Đúng như mục 6: `x = min(nums)`, `y` = giá trị nhỏ nhất có tần suất khác `cnt[x]`.

In [11]:
class SolutionFirstDiff:
    def minDistinctFreqPair(self, nums):
        cnt = Counter(nums)
        vals = sorted(cnt)
        x = vals[0]
        for y in vals[1:]:
            if cnt[y] != cnt[x]:
                return [x, y]
        return [-1, -1]

print(SolutionFirstDiff().minDistinctFreqPair([1, 1, 2, 2, 3, 4]), SolutionFirstDiff().minDistinctFreqPair([1, 5]), SolutionFirstDiff().minDistinctFreqPair([7]))

[1, 3] [-1, -1] [-1, -1]


**Độ phức tạp:** Time `O(n + m log m)` (sắp xếp giá trị phân biệt), Space `O(m)`.

### Cách 5 — Mảng đếm kích thước 101

Vì giá trị `<= 100`, dùng `freq[0..100]`; không cần `sorted` vì quét `v = 1..100` đã tăng dần.

In [12]:
class SolutionArray:
    def minDistinctFreqPair(self, nums):
        freq = [0] * 101
        for v in nums:
            freq[v] += 1
        x = next(v for v in range(1, 101) if freq[v])         # giá trị nhỏ nhất
        for y in range(x + 1, 101):
            if freq[y] and freq[y] != freq[x]:
                return [x, y]
        return [-1, -1]

print(SolutionArray().minDistinctFreqPair([1, 1, 2, 2, 3, 4]), SolutionArray().minDistinctFreqPair([1, 5]), SolutionArray().minDistinctFreqPair([7]))

[1, 3] [-1, -1] [-1, -1]


**Độ phức tạp:** Time `O(n + 100)`, Space `O(101)` hằng số.

## 8. TRACE từng bước

Trace Cách 4 trên `nums = [1,1,2,2,3,4]` và trên `[5,5,9,9]` (vô nghiệm):

In [13]:
def trace_first_diff(nums):
    cnt = Counter(nums)
    vals = sorted(cnt)
    x = vals[0]
    print(f"cnt = {dict(sorted(cnt.items()))}, x = {x} (tần suất {cnt[x]})")
    for y in vals[1:]:
        same = cnt[y] == cnt[x]
        print(f"  thử y={y}: tần suất {cnt[y]} {'==' if same else '!='} {cnt[x]} -> {'bỏ qua' if same else 'NHẬN'}")
        if not same:
            return [x, y]
    print("  không có y nào")
    return [-1, -1]

assert trace_first_diff([1, 1, 2, 2, 3, 4]) == [1, 3]
assert trace_first_diff([5, 5, 9, 9]) == [-1, -1]
assert trace_first_diff([1, 5]) == [-1, -1]

cnt = {1: 2, 2: 2, 3: 1, 4: 1}, x = 1 (tần suất 2)
  thử y=2: tần suất 2 == 2 -> bỏ qua
  thử y=3: tần suất 1 != 2 -> NHẬN
cnt = {5: 2, 9: 2}, x = 5 (tần suất 2)
  thử y=9: tần suất 2 == 2 -> bỏ qua
  không có y nào
cnt = {1: 1, 5: 1}, x = 1 (tần suất 1)
  thử y=5: tần suất 1 == 1 -> bỏ qua
  không có y nào


In [14]:
def trace_combos(nums):
    cnt = Counter(nums)
    for x, y in combinations(sorted(cnt), 2):
        print(f"cặp ({x},{y}): {cnt[x]} vs {cnt[y]}")
        if cnt[x] != cnt[y]:
            print("-> trả về", [x, y])
            return [x, y]
    return [-1, -1]

trace_combos([3, 3, 1, 1, 2])

cặp (1,2): 2 vs 1
-> trả về [1, 2]


[1, 2]

## 9. Code nộp LeetCode

Dùng Cách 4 (so với giá trị nhỏ nhất).

In [15]:
from typing import List, Optional
from collections import Counter


class Solution:
    def minDistinctFreqPair(self, nums: List[int]) -> List[int]:
        cnt = Counter(nums)
        vals = sorted(cnt)
        x = vals[0]                      # nếu có đáp án thì x luôn là giá trị nhỏ nhất
        for y in vals[1:]:
            if cnt[y] != cnt[x]:
                return [x, y]
        return [-1, -1]


print(Solution().minDistinctFreqPair([1, 1, 2, 2, 3, 4]), Solution().minDistinctFreqPair([1, 5]), Solution().minDistinctFreqPair([7]))

[1, 3] [-1, -1] [-1, -1]


## 10. Vì sao thuật toán đúng

1. **Nếu không có đáp án.** Khi mọi giá trị phân biệt cùng tần suất `f0`, không cặp nào có tần suất khác nhau, nên `[-1, -1]` đúng. Vòng lặp không tìm được `y` nào và trả về đúng `[-1, -1]`.
2. **Nếu có đáp án, `x = v0`.** Có ít nhất một giá trị `v` với `cnt[v] != cnt[v0]`; vì `v0` là nhỏ nhất nên `v0 < v` và `(v0, v)` hợp lệ. Mọi cặp hợp lệ khác có `x' >= v0` (vì `v0` là giá trị nhỏ nhất), do đó `x = v0` là nhỏ nhất.
3. **Chọn `y`.** Trong các cặp có `x = v0`, ta cần `y` nhỏ nhất thỏa `cnt[y] != cnt[v0]`. Duyệt `vals` tăng dần và dừng ở `y` đầu tiên như vậy cho đúng giá trị đó.
4. Các cách 1, 2, 3, 5 duyệt cặp theo thứ tự từ điển (hoặc lấy `min` theo tuple) nên cho cùng đáp án.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)`, `m` là số giá trị phân biệt (`m <= min(n, 100)`).

- Cách 1: `O(100² · n)` ở xấu nhất; Space `O(1)`.
- Cách 2, 3: `O(n + m²)` thời gian; Cách 3 thêm `O(m²)` bộ nhớ cho list cặp.
- Cách 4: `O(n + m log m)` thời gian, `O(m)` bộ nhớ.
- Cách 5: `O(n + 100)` thời gian, `O(101)` bộ nhớ.

Average và worst khác nhau ở cách 1, 2: nếu cặp đáp án xuất hiện sớm thì kết thúc sớm; trường hợp xấu nhất là vô nghiệm (phải duyệt hết mọi cặp).

## 12. Bộ test

Đối chiếu 5 cách và code nộp với ví dụ đề, biên và 300 input ngẫu nhiên đúng constraints.

In [16]:
import random

solutions = [SolutionBrute(), SolutionCombos(), SolutionMinTuple(), SolutionFirstDiff(), SolutionArray(), Solution()]

fixed = [
    ([1, 1, 2, 2, 3, 4], [1, 3]),
    ([1, 5], [-1, -1]),
    ([7], [-1, -1]),
    ([2, 2, 2], [-1, -1]),
    ([5, 5, 9, 9], [-1, -1]),
    ([3, 3, 1, 1, 2], [1, 2]),
    ([4, 4, 4, 1], [1, 4]),             # x = 1 (nhỏ nhất), y = 4
    ([1, 2, 2, 3, 3], [1, 2]),
    ([10, 20, 30, 30], [10, 30]),
    ([100, 100, 1, 1, 1], [1, 100]),
]
for nums, expected in fixed:
    for s in solutions:
        assert s.minDistinctFreqPair(list(nums)) == expected, (type(s).__name__, nums)

random.seed(38520)
for _ in range(300):
    n = random.randint(1, 100)
    hi = random.choice([2, 4, 10, 100])
    nums = [random.randint(1, hi) for _ in range(n)]
    outs = [s.minDistinctFreqPair(list(nums)) for s in solutions]
    assert all(o == outs[0] for o in outs), (nums, outs)
    x, y = outs[0]
    if x != -1:
        assert x < y and nums.count(x) != nums.count(y)
print("tất cả test OK")

tất cả test OK


### Test thuộc tính

Nếu trả về `[x, y]` thì `x = min(nums)`; thay đổi thứ tự `nums` không đổi đáp án; nhân đôi mọi phần tử (mỗi tần suất nhân 2) không đổi đáp án.

In [17]:
random.seed(3852)
for _ in range(300):
    n = random.randint(1, 50)
    nums = [random.randint(1, random.choice([3, 10, 100])) for _ in range(n)]
    ans = Solution().minDistinctFreqPair(nums)
    if ans != [-1, -1]:
        assert ans[0] == min(nums)
    shuf = nums[:]
    random.shuffle(shuf)
    assert Solution().minDistinctFreqPair(shuf) == ans
    assert Solution().minDistinctFreqPair(nums + nums) == ans
print("property test OK")

property test OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: chỉ xét cặp giá trị **kề nhau**

Nhiều bạn so sánh `vals[i]` với `vals[i + 1]` thay vì với mọi `y > x`. Với `[1,1,2,2,3,4]` cặp kề nhau đầu tiên khác tần suất là `(2, 3)`, không phải `(1, 3)`.

In [18]:
def wrong_adjacent(nums):
    cnt = Counter(nums)
    vals = sorted(cnt)
    for i in range(len(vals) - 1):
        if cnt[vals[i]] != cnt[vals[i + 1]]:           # SAI: chỉ so sánh hai giá trị kề nhau
            return [vals[i], vals[i + 1]]
    return [-1, -1]

print(wrong_adjacent([1, 1, 2, 2, 3, 4]), "!=", Solution().minDistinctFreqPair([1, 1, 2, 2, 3, 4]))
assert wrong_adjacent([1, 1, 2, 2, 3, 4]) == [2, 3] and Solution().minDistinctFreqPair([1, 1, 2, 2, 3, 4]) == [1, 3]

[2, 3] != [1, 3]


### Lỗi 2: so sánh giá trị thay vì tần suất

Điều kiện `x != y` luôn đúng với hai giá trị phân biệt, nên mọi cặp đều "hợp lệ" và ta luôn trả về hai giá trị nhỏ nhất.

In [19]:
def wrong_compare_values(nums):
    vals = sorted(set(nums))
    for x, y in combinations(vals, 2):
        if x != y:                                      # SAI: phải so cnt[x] != cnt[y]
            return [x, y]
    return [-1, -1]

print(wrong_compare_values([1, 5]), "!=", Solution().minDistinctFreqPair([1, 5]))
assert wrong_compare_values([1, 5]) == [1, 5] and Solution().minDistinctFreqPair([1, 5]) == [-1, -1]

[1, 5] != [-1, -1]


### Lỗi 3: không sắp xếp các giá trị (duyệt `Counter` theo thứ tự xuất hiện)

`Counter` giữ thứ tự gặp lần đầu. Nếu `nums = [3, 3, 1, 1, 2]` thì thứ tự là `3, 1, 2` và cặp đầu tiên khác tần suất là `(1, 2)`; nếu `nums = [2, 1, 1]` thì ta nhận `[2, 1]` (x > y) thay vì `[1, 2]`.

In [20]:
def wrong_unsorted(nums):
    cnt = Counter(nums)
    vals = list(cnt)                                    # SAI: thiếu sorted
    for x, y in combinations(vals, 2):
        if cnt[x] != cnt[y]:
            return [x, y]
    return [-1, -1]

print(wrong_unsorted([2, 1, 1]), "!=", Solution().minDistinctFreqPair([2, 1, 1]))
assert wrong_unsorted([2, 1, 1]) == [2, 1] and Solution().minDistinctFreqPair([2, 1, 1]) == [1, 2]

[2, 1] != [1, 2]


### Lỗi 4: dùng `y` bất kỳ thay vì `y` nhỏ nhất

Lấy giá trị **lớn nhất** (hoặc phần tử cuối) có tần suất khác ra cặp không nhỏ nhất.

In [21]:
def wrong_largest_y(nums):
    cnt = Counter(nums)
    vals = sorted(cnt)
    x = vals[0]
    best = None
    for y in vals[1:]:
        if cnt[y] != cnt[x]:
            best = y                                    # SAI: không dừng ở y đầu tiên
    return [x, best] if best is not None else [-1, -1]

print(wrong_largest_y([1, 1, 2, 3]), "!=", Solution().minDistinctFreqPair([1, 1, 2, 3]))
assert wrong_largest_y([1, 1, 2, 3]) == [1, 3] and Solution().minDistinctFreqPair([1, 1, 2, 3]) == [1, 2]

[1, 3] != [1, 2]


### Lỗi 5: quên trường hợp chỉ có một giá trị phân biệt

`vals[1:]` rỗng thì không có `y`; nếu code truy cập `vals[1]` trực tiếp sẽ `IndexError`.

In [22]:
def wrong_index(nums):
    cnt = Counter(nums)
    vals = sorted(cnt)
    x, y = vals[0], vals[1]                              # SAI: không có vals[1] khi chỉ có 1 giá trị
    return [x, y] if cnt[x] != cnt[y] else [-1, -1]

try:
    wrong_index([7])
except IndexError as e:
    print("IndexError:", e)
assert Solution().minDistinctFreqPair([7]) == [-1, -1]

IndexError: list index out of range


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Tần suất" + "giá trị phân biệt" -> `Counter`, rồi quên mảng gốc đi, chỉ làm việc trên bảng `giá trị -> tần suất`.
- "Nhỏ nhất theo thứ tự từ điển" -> sắp xếp rồi duyệt tăng dần và dừng ở lần đầu thỏa, hoặc dùng `min` trên tuple.
- Với ràng buộc nhỏ (`<= 100`) mảng đếm `freq[0..100]` vừa nhanh vừa đơn giản.
- Khi thấy brute force `O(m²)` hãy tự hỏi "nếu có đáp án thì phần tử nào chắc chắn tham gia?" (ở đây là `min`).
- Luôn thử các trường hợp: một giá trị, mọi giá trị cùng tần suất, giá trị nhỏ nhất là giá trị "lạ".

## 15. Biến thể của bài

1. **First Unique Number / First Unique Character** (387): giá trị xuất hiện đúng một lần.
2. **Majority Element** (169), **Top K Frequent Elements** (347): xếp hạng theo tần suất.
3. **Unique Number of Occurrences** (1207): các tần suất đôi một khác nhau.
4. **Sort Characters By Frequency** (451): sắp xếp theo tần suất.
5. Biến thể: chọn cặp có `y` nhỏ nhất rồi mới `x`, hoặc cặp có **hiệu `y - x`** nhỏ nhất.
6. Biến thể: yêu cầu tần suất **bằng nhau** thay vì khác nhau (đòi hỏi nhóm theo tần suất).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute | `count` với mọi cặp giá trị | `O(100² · n)` | `O(1)` | Chỉ để kiểm tra |
| 2. Combinations | cặp từ điển trên giá trị đã sắp xếp | `O(n + m²)` | `O(m)` | Dễ đọc, sát đề |
| 3. `min` tuple | gom mọi cặp rồi lấy min | `O(n + m²)` | `O(m²)` | Oracle, ít sai sót |
| 4. So với giá trị nhỏ nhất | insight `x = min` | `O(n + m log m)` | `O(m)` | Nộp bài |
| 5. Mảng đếm | `freq[0..100]` | `O(n + 100)` | `O(101)` | Khi giá trị nhỏ, không cần `sorted` |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().minDistinctFreqPair([2, 2, 3])` và `Solution().minDistinctFreqPair([4, 4, 5, 5])` ra gì?

**Bài 2 (tính tay):** với `nums = [6, 6, 6, 2, 9, 9]`, lập bảng tần suất và xác định cặp đáp án.

**Bài 3 (điền TODO):** hoàn thành `freq_table(nums)` trả về list các cặp `(giá_trị, tần_suất)` sắp xếp theo giá trị tăng dần.

**Bài 4 (tự viết lại):** viết `has_equal_frequency_pair(nums)` trả về `True` nếu tồn tại hai giá trị phân biệt có **cùng** tần suất.

**Bài 5 (mini challenge):** viết `largest_pair_diff_freq(nums)` trả về cặp `[x, y]` có `x < y` khác tần suất sao cho `x` **lớn nhất**, rồi `y` lớn nhất (đối xứng với đề gốc), hoặc `[-1, -1]`.

In [23]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(Solution().minDistinctFreqPair([2, 2, 3]), Solution().minDistinctFreqPair([4, 4, 5, 5]))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def freq_table_todo(nums):
    result = []
    # TODO: dùng Counter rồi sắp xếp theo giá trị
    return result

[2, 3] [-1, -1]


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** Bài 1: `[2,2,3]` có `2` xuất hiện 2 lần, `3` xuất hiện 1 lần; `[4,4,5,5]` cả hai 2 lần. Bài 2: `6` x3, `2` x1, `9` x2.

**Mức 2 (vừa):** Bài 3: `sorted(Counter(nums).items())`. Bài 4: đếm số lần của từng **tần suất**: nếu tần suất nào xuất hiện từ 2 lần trở lên thì có cặp. Bài 5: duyệt giá trị giảm dần; cẩn thận `x < y`.

**Mức 3 (gần đáp án):** Bài 1: `[2,3]` và `[-1,-1]`. Bài 2: `x = 2` (nhỏ nhất), `y` = giá trị nhỏ nhất `> 2` khác tần suất 1: `6` có 3 lần -> `[2,6]`. Bài 5: với `[1,1,2,3]`, `x = 2` thì `y = 3` cùng tần suất 1 nên không hợp lệ; thử tiếp `x = 1`: `y = 3` hợp lệ (2 so với 1), nên đáp án `[1, 3]`.

## 19. Đáp án tham khảo cho bài tập

In [24]:
# Bài 1, 2
assert Solution().minDistinctFreqPair([2, 2, 3]) == [2, 3]
assert Solution().minDistinctFreqPair([4, 4, 5, 5]) == [-1, -1]
assert Solution().minDistinctFreqPair([6, 6, 6, 2, 9, 9]) == [2, 6]

# Bài 3
def freq_table(nums):
    return sorted(Counter(nums).items())
assert freq_table([6, 6, 6, 2, 9, 9]) == [(2, 1), (6, 3), (9, 2)]

# Bài 4
def has_equal_frequency_pair(nums):
    freqs = list(Counter(nums).values())
    return len(freqs) != len(set(freqs))
assert has_equal_frequency_pair([1, 1, 2, 2]) is True
assert has_equal_frequency_pair([1, 2, 2, 3, 3, 3]) is False and has_equal_frequency_pair([5]) is False

# Bài 5
def largest_pair_diff_freq(nums):
    cnt = Counter(nums)
    vals = sorted(cnt, reverse=True)
    for x in vals:
        bigger = [y for y in sorted(cnt) if y > x and cnt[y] != cnt[x]]
        if bigger:
            return [x, bigger[-1]]
    return [-1, -1]
assert largest_pair_diff_freq([1, 1, 2, 3]) == [1, 3]
assert largest_pair_diff_freq([4, 4, 5, 5]) == [-1, -1]
assert largest_pair_diff_freq([1, 2, 2, 3, 3, 3]) == [2, 3]
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "lấy bảng tần suất, rồi chọn theo thứ tự từ điển":**

```python
cnt = Counter(nums)
vals = sorted(cnt)
x = vals[0]                       # x nhỏ nhất (nếu có đáp án)
for y in vals[1:]:
    if cnt[y] != cnt[x]:
        return [x, y]
return [-1, -1]
```

**Checklist trước khi submit:**

- [ ] Đang so sánh **tần suất** (không phải giá trị)?
- [ ] `x < y` và duyệt giá trị theo thứ tự tăng dần (đã `sorted`)?
- [ ] Chọn `x` nhỏ nhất rồi mới `y` nhỏ nhất (dừng ở `y` đầu tiên thỏa)?
- [ ] Đã xử lý trường hợp chỉ có một giá trị / mọi giá trị cùng tần suất -> `[-1, -1]`?
- [ ] Đã thử `[1,1,2,2,3,4]`, `[1,5]`, `[7]`?
- [ ] Chữ ký đúng `minDistinctFreqPair(self, nums: List[int]) -> List[int]`?